#### **OPTIMALIDAD DE LA SOLUCION - ALGORITMOS GREEDY**

In [45]:
import pandas as pd 
from importlib import reload
import random

import Clases.asignacion as asignacion_module
reload(asignacion_module)
from Clases.asignacion import Asignacion

import Clases.caja as caja_module
reload(caja_module)
from Clases.caja import Caja

import Clases.producto as producto_module
reload(producto_module)
from Clases.producto import Producto

import Clases.solucion as solucion_module
reload(solucion_module)
from Clases.solucion import Solucion

import Algoritmos.greedy as greedy_module
reload(greedy_module)
from Algoritmos.greedy import solucion_greedy

import Algoritmos.relocate as relocate_module
reload(relocate_module)
from Algoritmos.relocate import relocate

import Algoritmos.swap as swap_module
reload(swap_module)
from Algoritmos.swap import swap

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")

cajas_nuevas = pd.read_csv("4r.cajas_nuevas2.csv")
factibilidad = pd.read_csv("Factibilidad2/factibilidad_3mm.csv")

In [46]:
grosor = 3

Empecemos guardando los productos y tipos de cajas en listas en el estado actual, para cargarlos luego a las soluciones. Almacenamos también las cajas asignables a cada producto en un diccionario.

In [47]:
def guardar_cajas_y_productos(grosor=3):
    
    cajas = {
    caja_id: Caja(caja_id=caja_id, dim_interior_ancho=ancho, dim_interior_largo=largo, dim_interior_alto=alto)
    for caja_id, ancho, largo, alto in zip(
        cajas_nuevas["caja_tipo_id"],
        cajas_nuevas["caja_interior_ancho"],
        cajas_nuevas["caja_interior_largo"],
        cajas_nuevas["caja_interior_alto"]
    )
}

    prod_op_merge = catalogo_productos.merge(operaciones_planta, on="codigo_producto")
    productos = {
    row.codigo_producto: Producto(
        codigo_producto=row.codigo_producto,
        cantidad_paquetes=row.cantidad_paquetes,
        peso_paquete=row.peso_neto_paquete,
        demanda_buenos_aires=row.volumen_producto_planta_buenos_aires,
        demanda_curitiba=row.volumen_producto_planta_curitiba,
        demanda_santiago=row.volumen_producto_planta_santiago,
        demanda_monterrey=row.volumen_producto_planta_monterrey,
        demanda_bakersfield=row.volumen_producto_planta_bakersfield,
        dim_producto_ancho=row.dim_producto_ancho,
        dim_producto_largo=row.dim_producto_largo,
        dim_producto_alto=row.dim_producto_alto
    )
    for row in prod_op_merge.itertuples(index=False)
}
    
    cajas_asignables_por_producto = {}

    for codigo, group in factibilidad.groupby('codigo_producto'):
        # Obtener IDs de los tipos de cajas
        cajas_ids_unicos = list(group['caja_tipo_id'].unique())
        
        cajas_producto = []
        for caja_id in cajas_ids_unicos:
            cajas_producto.append(caja_id)
            
        cajas_asignables_por_producto[codigo] = cajas_producto
                
    # Elegir grosor
    for caja_id, caja in cajas.items():
        caja.elegir_grosor(grosor_mm=grosor)
        
    return cajas, productos, cajas_asignables_por_producto

In [48]:
def ordenar_por_cajas_asignables(asignaciones_por_producto):
    '''
    Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
    '''
    productos_conteo = []

    for codigo_producto, cajas_asignables in asignaciones_por_producto.items():
        cantidad_cajas = len(cajas_asignables)  # Número de cajas asignables para este producto    
        productos_conteo.append({
            'codigo_producto': codigo_producto,
            'cantidad_cajas_asignables': cantidad_cajas
        })

    # Ordenar de menor a mayor cantidad de cajas
    productos_ordenados = sorted(
        productos_conteo,
        key=lambda x: x['cantidad_cajas_asignables'],
        reverse=False
    )

    lista_productos_ordenados = [item['codigo_producto'] for item in productos_ordenados]
    return lista_productos_ordenados

In [49]:
cajas, productos, asignaciones_por_producto = guardar_cajas_y_productos()
lista_productos_ordenados = ordenar_por_cajas_asignables(asignaciones_por_producto)

solucion_greedy = solucion_greedy(cajas, productos, lista_productos_ordenados, asignaciones_por_producto,
                                  criterio_greedy="maximizar_utilizacion_pallet",
                                  titulo_solucion="Greedy maximizando utilizacion pallet | Ordenamiento según #cajas asignables",
                                  grosor=grosor)

In [50]:
solucion_greedy.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30166293.939999998
Costo flete: 179068800
Costo total: 209235093.94
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Criterio elegido: Greedy maximizando utilizacion pallet | Ordenamiento según #cajas asignables
Número de tipos de cajas distintos: 76
Costo packaging: 27406334.33999999
Costo flete: 161380650
Costo total: 188786984.33999997
Utilización de pallet promedio: 0.9593515126880586
Utilización de caja promedio: 0.9445527162171277
Ahorro costo total: 9.77279%


In [51]:
relocate(solucion_greedy, cajas, productos, asignaciones_por_producto, 
         titulo_solucion="Greedy + Relocate maximizando utilizacion pallet")

Baja de  188786984.33999997 a  188785038.0
Baja de  188785038.0 a  188783643.35999998
Baja de  188783643.35999998 a  188780265.48
Baja de  188780265.48 a  188775808.98
Baja de  188775808.98 a  188772792.0
Baja de  188772792.0 a  188772466.98
Baja de  188772466.98 a  188763002.1
Baja de  188763002.1 a  188761265.16
Baja de  188761265.16 a  188759516.45999998
Baja de  188759516.45999998 a  188759506.26
Baja de  188759506.26 a  188759367.06
Baja de  188759367.06 a  188759364.95999998
Baja de  188759364.95999998 a  188759111.94
Baja de  188759111.94 a  188756531.57999998
Baja de  188756531.57999998 a  188756159.1
Baja de  188756159.1 a  188753083.01999998
Baja de  188753083.01999998 a  188752135.98
Baja de  188752135.98 a  188745766.85999998
Baja de  188745766.85999998 a  188743681.79999998
Baja de  188743681.79999998 a  188742214.56
Baja de  188742214.56 a  188741883.89999998
Baja de  188741883.89999998 a  188741874.66
Baja de  188741874.66 a  188724437.16
Baja de  188724437.16 a  1887171

In [52]:
solucion_greedy.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30166293.939999998
Costo flete: 179068800
Costo total: 209235093.94
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Criterio elegido: Greedy + Relocate maximizando utilizacion pallet
Número de tipos de cajas distintos: 56
Costo packaging: 27039692.099999994
Costo flete: 161244750
Costo total: 188284442.1
Utilización de pallet promedio: 0.9522236587027646
Utilización de caja promedio: 0.9530217053134908
Ahorro costo total: 10.01297%


In [53]:
swap(solucion_greedy, productos, asignaciones_por_producto, 
    titulo_solucion="Greedy + Swap maximizando utilizacion pallet")

Swap mejoró de 188284442.1 a 188282507.94


In [54]:
solucion_greedy.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30166293.939999998
Costo flete: 179068800
Costo total: 209235093.94
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Criterio elegido: Greedy + Swap maximizando utilizacion pallet
Número de tipos de cajas distintos: 56
Costo packaging: 27037757.939999998
Costo flete: 161244750
Costo total: 188282507.94
Utilización de pallet promedio: 0.9522408730195074
Utilización de caja promedio: 0.9530050227499599
Ahorro costo total: 10.01390%


In [55]:
relocate(solucion_greedy, cajas, productos, asignaciones_por_producto, 
         titulo_solucion="Greedy + Relocate maximizando utilizacion pallet")

In [56]:
solucion_greedy.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30166293.939999998
Costo flete: 179068800
Costo total: 209235093.94
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 3mm
Criterio elegido: Greedy + Relocate maximizando utilizacion pallet
Número de tipos de cajas distintos: 56
Costo packaging: 27037757.939999998
Costo flete: 161244750
Costo total: 188282507.94
Utilización de pallet promedio: 0.9522408730195074
Utilización de caja promedio: 0.9530050227499599
Ahorro costo total: 10.01390%
